In [1]:
import csv
import pandas as pd

In [2]:
RESULTS = {
    1:  "../scenarios/runs/2026-10-19/110307_1GB_full",
    2:  "../scenarios/runs/2026-10-20/095044_2GB_full",
    5:  "../scenarios/runs/2026-10-20/101602_5GB_full",
    10: "../scenarios/runs/2026-10-21/173743_10GB_full"
}

QUERY_RUNS_FILENAME = "query_runs.csv"

In [3]:
TOP_N = 10

def load_query_times(run_dir):
    """Seconds per query, summing the statement parts of multi-statement queries
    (14, 23, 24, 39)."""
    runs = pd.read_csv(f"{run_dir}/{QUERY_RUNS_FILENAME}")
    return runs.groupby("query_id")["avg_seconds"].sum()


query_times = pd.DataFrame(
    {f"{scale}GB": load_query_times(run_dir) for scale, run_dir in RESULTS.items()}
)
query_times.sum().rename("total_seconds")

1GB     151.587334
2GB     214.670867
5GB     370.500640
10GB    573.881465
Name: total_seconds, dtype: float64

## Slowest queries per scale factor

In [4]:
def top_queries(times, n=TOP_N):
    return [
        f"q{query_id} – {seconds:.2f}"
        for query_id, seconds in times.nlargest(n).items()
    ]


pd.DataFrame(
    {scale: top_queries(query_times[scale]) for scale in query_times},
    index=pd.RangeIndex(1, TOP_N + 1, name="rank"),
)

,1GB,2GB,5GB,10GB
rank,,,,
1,q14 – 9.99,q23 – 16.38,q23 – 40.04,q72 – 61.27
2,q23 – 9.80,q72 – 14.62,q72 – 27.03,q23 – 50.67
3,q72 – 8.19,q14 – 13.20,q14 – 25.21,q67 – 27.69
4,q4 – 4.53,q67 – 8.24,q4 – 20.43,q14 – 27.55
5,q67 – 4.29,q95 – 7.93,q17 – 15.70,q4 – 24.50
6,q22 – 4.16,q4 – 6.73,q67 – 15.19,q24 – 22.14
7,q95 – 4.07,q22 – 6.55,q22 – 15.03,q64 – 22.01
8,q47 – 3.76,q47 – 5.24,q64 – 12.19,q78 – 20.83
9,q10 – 3.05,q11 – 4.07,q11 – 10.99,q80 – 20.44


## How the slowest queries scale

Every query that is in the top N at any scale, with its time at each scale, how many times slower it is at the largest scale than at the smallest (`growth`), and its share of the largest run.

In [5]:
slowest_ids = sorted(
    set().union(*(query_times[scale].nlargest(TOP_N).index for scale in query_times))
)
smallest, largest = query_times.columns[[0, -1]]

growth = (
    query_times.loc[slowest_ids]
    .assign(
        growth=lambda df: df[largest] / df[smallest],
        share_of_largest_run=lambda df: df[largest] / query_times[largest].sum(),
    )
    .sort_values(largest, ascending=False)
)
growth.round(2)

,1GB,2GB,5GB,10GB,growth,share_of_largest_run
query_id,,,,,,
72,8.19,14.62,27.03,61.27,7.48,0.11
23,9.80,16.38,40.04,50.67,5.17,0.09
67,4.29,8.24,15.19,27.69,6.45,0.05
14,9.99,13.20,25.21,27.55,2.76,0.05
4,4.53,6.73,20.43,24.50,5.40,0.04
24,1.59,2.53,10.69,22.14,13.90,0.04
64,2.53,3.86,12.19,22.01,8.69,0.04
78,2.58,3.95,9.43,20.83,8.08,0.04
80,2.02,2.29,8.12,20.44,10.15,0.04
